# Diagnóstico: `_normalize_timestamps` do HoTHP

**Hipótese investigada:** a função `_normalize_timestamps` cancela algebricamente os timestamps reais,
produzindo sempre `[0, 1, 2, ..., T-1]` — independente dos tempos de chegada dos eventos.

Se confirmado, o kernel hiperbólico do HoTHP não usa informação temporal real;
usa apenas distância em índice de evento (posição).

**Pipeline exato reproduzido:**
1. Simular sequências de Hawkes
2. `to_tensors()` — normaliza por gap médio (como nos experimentos)
3. `_normalize_timestamps()` — normalização adicional do HoTHP
4. Comparar com o que o RoTHP recebe (só o passo 2)


In [1]:
import sys, os
import numpy as np
import torch

# Ajusta path para encontrar easy_tpp
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print('Python path OK')

Python path OK


## 1. Simular sequências de Hawkes (processo de decaimento rápido)

In [2]:
def simulate_hawkes(rng, mu, alpha, beta, horizon, min_ev, max_ev):
    """Simula um processo de Hawkes multivariado pelo método de thinning."""
    for _ in range(100):
        events, t = [], 0.0
        while t < horizon and len(events) < max_ev:
            lam = mu.copy()
            for ti, ki in events:
                lam += alpha[:, ki] * np.exp(-beta * (t - ti))
            lam_bar = float(lam.sum())
            if lam_bar < 1e-9:
                break
            t += rng.exponential(1.0 / lam_bar)
            if t >= horizon:
                break
            cand = mu.copy()
            for ti, ki in events:
                cand += alpha[:, ki] * np.exp(-beta * (t - ti))
            if rng.uniform() <= cand.sum() / lam_bar:
                probs = cand / cand.sum()
                events.append((t, int(rng.choice(len(mu), p=probs))))
        if len(events) >= min_ev:
            return events[:max_ev]
    return events[:max_ev]


# Parâmetros: decaimento rápido (mesmo dos experimentos)
mu    = np.array([0.4, 0.4])
alpha = np.array([[0.12, 0.08], [0.08, 0.12]])
beta  = 0.50

rng = np.random.default_rng(42)
raw_seqs = [simulate_hawkes(rng, mu, alpha, beta, 50.0, 5, 20) for _ in range(10)]

print('Sequências simuladas (timestamps brutos):') 
print('=' * 70)
for i, seq in enumerate(raw_seqs):
    times = [round(t, 4) for t, k in seq]
    print(f'Seq {i+1:2d} (N={len(seq):2d}): {times[:8]}...')

Sequências simuladas (timestamps brutos):
Seq  1 (N=20): [3.0053, 3.2851, 4.4862, 5.3546, 7.9184, 8.1977, 9.1681, 10.1128]...
Seq  2 (N=20): [1.8848, 3.5296, 3.5705, 5.0577, 7.1351, 7.7888, 7.896, 7.9342]...
Seq  3 (N=20): [0.0882, 0.3825, 1.4029, 1.479, 2.4718, 2.6808, 3.1955, 3.4999]...
Seq  4 (N=20): [3.0506, 4.0998, 4.3447, 4.546, 5.3045, 5.6098, 5.6788, 7.3475]...
Seq  5 (N=20): [3.3374, 3.8507, 3.9017, 4.2518, 6.369, 6.5324, 6.6169, 7.6457]...
Seq  6 (N=20): [1.2826, 1.8825, 2.2363, 2.7011, 3.3875, 3.7142, 3.8694, 4.5357]...
Seq  7 (N=20): [3.8838, 4.2073, 6.0595, 7.0149, 7.2793, 7.4198, 7.4282, 8.7481]...
Seq  8 (N=20): [1.8557, 2.3309, 2.4406, 2.7059, 3.6511, 6.039, 8.7641, 9.0463]...
Seq  9 (N=20): [0.8935, 2.0726, 2.0903, 2.2403, 2.6561, 4.6715, 5.093, 5.4046]...
Seq 10 (N=20): [1.014, 2.0802, 3.2482, 3.3708, 3.4889, 3.8661, 4.3522, 4.3703]...


## 2. `to_tensors()` — pipeline dos experimentos

Normaliza por gap médio, igual ao que os notebooks de experimento fazem.

In [3]:
def to_tensors(seqs):
    """Reproduz exatamente a função to_tensors dos experimentos."""
    out = []
    for seq in seqs:
        seq = sorted(seq, key=lambda x: x[0])
        t = torch.tensor([x[0] for x in seq], dtype=torch.float32)
        k = torch.tensor([x[1] for x in seq], dtype=torch.long)
        d = torch.zeros_like(t)
        d[1:] = t[1:] - t[:-1]
        mg = d[1:].mean().clamp(min=1e-6)   # gap médio (causal: ignora d[0]=0)
        t = (t - t[0]) / mg                 # normaliza por gap médio
        d = d / mg
        out.append({'time_seqs': t, 'time_delta_seqs': d, 'type_seqs': k})
    return out


tensors = to_tensors(raw_seqs)

print('Após to_tensors() — o que RoTHP recebe como time_seqs:')
print('(timestamps normalizados por gap médio da sequência)')
print('=' * 70)
for i, item in enumerate(tensors):
    t = item['time_seqs']
    vals = [round(v, 4) for v in t[:8].tolist()]
    print(f'Seq {i+1:2d}: {vals}...')

Após to_tensors() — o que RoTHP recebe como time_seqs:
(timestamps normalizados por gap médio da sequência)
Seq  1: [0.0, 0.43, 2.2757, 3.6103, 7.5502, 7.9795, 9.4706, 10.9224]...
Seq  2: [0.0, 2.5535, 2.6169, 4.9257, 8.1506, 9.1654, 9.3319, 9.3912]...
Seq  3: [0.0, 0.3563, 1.5915, 1.6837, 2.8855, 3.1384, 3.7615, 4.13]...
Seq  4: [0.0, 1.4832, 1.8293, 2.114, 3.1862, 3.6177, 3.7152, 6.0743]...
Seq  5: [0.0, 0.767, 0.8433, 1.3663, 4.5299, 4.774, 4.9003, 6.4375]...
Seq  6: [0.0, 0.7635, 1.2137, 1.8051, 2.6786, 3.0944, 3.2919, 4.1398]...
Seq  7: [0.0, 0.3667, 2.4666, 3.5497, 3.8494, 4.0088, 4.0182, 5.5147]...
Seq  8: [0.0, 0.8899, 1.0952, 1.5919, 3.3617, 7.8327, 12.9352, 13.4636]...
Seq  9: [0.0, 1.5061, 1.5286, 1.7203, 2.2515, 4.8258, 5.3642, 5.7622]...
Seq 10: [0.0, 0.9993, 2.094, 2.2089, 2.3196, 2.6731, 3.1287, 3.1456]...


## 3. `_normalize_timestamps()` — o que HoTHP aplica em seguida

In [4]:
def _normalize_timestamps(time_seqs):
    """Cópia exata de HoTHP._normalize_timestamps (torch_hothp.py linha 209)."""
    B, T = time_seqs.shape
    t_shifted = time_seqs - time_seqs[:, :1]

    if T <= 1:
        return t_shifted

    diffs    = t_shifted[:, 1:] - t_shifted[:, :-1]
    cumsum   = torch.cumsum(diffs, dim=-1)
    counts   = torch.arange(1, T, dtype=time_seqs.dtype).unsqueeze(0)
    prefix_mean = cumsum / counts

    ones    = torch.ones(B, 1, dtype=time_seqs.dtype)
    divisor = torch.cat([ones, prefix_mean], dim=-1).clamp(min=1e-6)

    return t_shifted / divisor


# Empilha as sequências numa batch de tamanho fixo (trunca para a menor)
min_len = min(item['time_seqs'].shape[0] for item in tensors)
batch_t = torch.stack([item['time_seqs'][:min_len] for item in tensors])  # [10, min_len]

norm = _normalize_timestamps(batch_t)

print('Após _normalize_timestamps() — o que HoTHP usa no kernel:')
print('=' * 70)
for i in range(len(tensors)):
    vals = [round(v, 4) for v in norm[i, :8].tolist()]
    print(f'Seq {i+1:2d}: {vals}...')

print()
print('Valores esperados se hipótese for verdadeira: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]')

Após _normalize_timestamps() — o que HoTHP usa no kernel:
Seq  1: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  2: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  3: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  4: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  5: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  6: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  7: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  8: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq  9: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...
Seq 10: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]...

Valores esperados se hipótese for verdadeira: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0]


## 4. Prova algébrica direta

In [5]:
print('Verificação: norm[i] == torch.arange(min_len) para todas as sequências?')
print('=' * 70)

expected = torch.arange(min_len, dtype=torch.float32).unsqueeze(0).expand(len(tensors), -1)
diff = (norm - expected).abs()

print(f'Erro máximo (todas as seqs, todas as posições): {diff.max().item():.2e}')
print(f'Erro médio : {diff.mean().item():.2e}')
print()

if diff.max().item() < 1e-4:
    print('✓ CONFIRMADO: _normalize_timestamps() produz sempre [0, 1, 2, ..., T-1]')
    print('  Os Δt no kernel hiperbólico são diferenças de ÍNDICE, não de TEMPO.')
else:
    print('✗ Hipótese NÃO confirmada — investigar.')

Verificação: norm[i] == torch.arange(min_len) para todas as sequências?
Erro máximo (todas as seqs, todas as posições): 9.54e-07
Erro médio : 9.54e-09

✓ CONFIRMADO: _normalize_timestamps() produz sempre [0, 1, 2, ..., T-1]
  Os Δt no kernel hiperbólico são diferenças de ÍNDICE, não de TEMPO.


## 5. Impacto: o que RoTHP vs HoTHP realmente vê como Δt

In [ ]:
print('Comparação dos Δt entre evento 5 e os anteriores (sequência 1):')
print('=' * 70)

t_rothp = batch_t[0]
t_hothp = norm[0]
i = 5

print(f'Timestamps que RoTHP usa: {[round(v,3) for v in t_rothp[:8].tolist()]}')
print(f'Timestamps que HoTHP usa: {[round(v,3) for v in t_hothp[:8].tolist()]}')
print()

col_r = "RoTHP Δt (tempo real)"
col_h = "HoTHP Δt (índice)"
print(f'  {"j":>4}  {col_r:>25}  {col_h:>20}')
print(f'  {"-"*4}  {"-"*25}  {"-"*20}')
for j in range(i):
    dt_rothp = (t_rothp[i] - t_rothp[j]).item()
    dt_hothp = (t_hothp[i] - t_hothp[j]).item()
    print(f'  {j:>4}  {dt_rothp:>25.4f}  {dt_hothp:>20.4f}')

print()
print('Observação: para HoTHP, Δt(i,j) = i - j SEMPRE (inteiro).')
print('Duas sequências com gaps temporais completamente diferentes')
print('produzem os MESMOS Δt no kernel hiperbólico.')

## 6. Demonstração com duas sequências de gaps muito diferentes

In [ ]:
# Sequência A: eventos muito próximos (gaps ~0.1)
seq_A = torch.tensor([0.0, 0.1, 0.2, 0.3, 0.4, 0.5])

# Sequência B: eventos muito espaçados (gaps ~100)
seq_B = torch.tensor([0.0, 100.0, 200.0, 300.0, 400.0, 500.0])

# Simula to_tensors: normaliza por gap médio
def normalize_by_mean_gap(t):
    d = t[1:] - t[:-1]
    mg = d.mean().clamp(min=1e-6)
    return (t - t[0]) / mg

t_A_rothp = normalize_by_mean_gap(seq_A)   # RoTHP vê isso
t_B_rothp = normalize_by_mean_gap(seq_B)   # RoTHP vê isso

batch_AB = torch.stack([t_A_rothp, t_B_rothp])
norm_AB  = _normalize_timestamps(batch_AB)  # HoTHP usa isso

print('Sequência A (gaps ~0.1):   eventos muito próximos')
print('Sequência B (gaps ~100):   eventos muito espaçados')
print()
print('O que RoTHP recebe:')
print(f'  Seq A: {t_A_rothp.tolist()}')
print(f'  Seq B: {t_B_rothp.tolist()}')
print()
print('O que HoTHP usa no kernel:')
print(f'  Seq A: {norm_AB[0].tolist()}')
print(f'  Seq B: {norm_AB[1].tolist()}')
print()
print('Conclusão: RoTHP diferencia as sequências. HoTHP não.')

## 7. Verifica com o código real do HoTHP (não a cópia)

In [ ]:
from easy_tpp.model.torch_model.torch_hothp import HoTHP
from easy_tpp.config_factory.model_config import ModelConfig

config = ModelConfig(**{
    'hidden_size': 32, 'num_layers': 2, 'num_heads': 2, 'dropout_rate': 0.1,
    'num_event_types': 2, 'num_event_types_pad': 3,
    'event_pad_index': 2, 'time_emb_size': 32, 'use_ln': True,
    'gpu': -1, 'model_id': 'Diag',
    'thinning': {'num_sample': 1, 'num_exp': 500, 'over_sample_rate': 5.0,
                 'patience_counter': 5, 'num_samples_boundary': 5,
                 'dtime_max': 5.0, 'num_step_gen': 1},
    'loss_integral_num_sample_per_step': 20,
    'use_mc_samples': False,
})

model = HoTHP(config)

# Intercepta _normalize_timestamps para capturar entrada e saída
captured = {}
_orig = model._normalize_timestamps
def _patched(time_seqs):
    result = _orig(time_seqs)
    captured['input']  = time_seqs.detach().clone()
    captured['output'] = result.detach().clone()
    return result
model._normalize_timestamps = _patched

# Roda um forward com batch de 3 sequências de comprimento 6
L = 6
t_input = batch_t[:3, :L]
k_input = torch.zeros(3, L, dtype=torch.long)
mask    = torch.triu(torch.ones(L, L, dtype=torch.bool), diagonal=1).unsqueeze(0).expand(3,-1,-1)

with torch.no_grad():
    model.eval()
    _ = model.forward(t_input, k_input, mask)

print('Entrada de _normalize_timestamps (time_seqs reais do pipeline):')
for i in range(3):
    print(f'  Seq {i+1}: {[round(v,4) for v in captured["input"][i].tolist()]}')

print()
print('Saída de _normalize_timestamps (o que chega ao kernel hiperbólico):')
for i in range(3):
    print(f'  Seq {i+1}: {[round(v,4) for v in captured["output"][i].tolist()]}')

print()
expected_row = list(range(L))
all_match = all(
    all(abs(captured['output'][i, j].item() - j) < 1e-3 for j in range(L))
    for i in range(3)
)
if all_match:
    print(f'✓ CONFIRMADO com código real: saída sempre = {expected_row}')
else:
    print('✗ Resultado inesperado — revisar hipótese.')

## 8. Resumo

| | RoTHP | HoTHP |
|---|---|---|  
| Δt no kernel | tᵢ - tⱼ (tempo real normalizado) | i - j (índice inteiro sempre) |
| Seq A vs Seq B | distingue | não distingue |
| Informação temporal | preservada | destruída |

**Consequência:** O kernel hiperbólico aprende decaimento por *número de eventos atrás*, não por *distância temporal*. A motivação do Processo de Hawkes (decaimento em função de Δt real) não é satisfeita.

**Próximo passo:** corrigir `_normalize_timestamps` para preservar a estrutura temporal relativa.